# HW5: Fixed-Income Valuation and First-Order Risk

Complete the five library TODOs, then run all cells.

In [ ]:
from typing import cast

import numpy as np
import pandas as pd
from IPython.display import display

from fixedincomelib import (
    qfCreateData1D,
    qfCreateDataCollection,
    qfCreateBuildMethod,
    qfCreateModelBuildMethodCollection,
    qfCreateModel,
    qfCreateProductFixedAccruedCashflow,
    qfCreateValuationParameters,
    qfCreateValuationParametersCollection,
    qfCreateValueReport,
)
from fixedincomelib.date import Date
from fixedincomelib.market.registries import IndexRegistry
from fixedincomelib.valuation import ValuationRequest
from fixedincomelib.yield_curve import YieldCurveModelComponent
from fixedincomelib.yield_curve.valuation_engine import ValuationEngineProductFixedAccrued

np.set_printoptions(precision=6, suppress=True)
pd.options.display.float_format = "{:,.6f}".format
pd.options.display.max_colwidth = None

## 1. Input data

Rates are annual decimals: `0.03 = 3%`. Edit the inputs here and rerun the following cells.
The funding component adds an IFR spread over SOFR.

In [ ]:
VALUE_DATE = "2026-01-02"
PAYMENT_DATE = Date("2026-10-02")
FUNDING_INDEX = "SOFR-1B-FLAT"

sofr_quotes = pd.DataFrame(
    {"values": [0.03, 0.04, 0.05]},
    index=pd.Index(["2026-04-02", "2027-01-02", "2028-01-02"], name="Node date"),
)
funding_quotes = pd.DataFrame(
    {"values": [0.0, 0.001]},
    index=pd.Index(["2026-07-02", "2027-01-02"], name="Node date"),
)

display(sofr_quotes.rename(columns={"values": "SOFR-1B IFR"}))
display(funding_quotes.rename(columns={"values": "SOFR-1B-FLAT IFR spread"}))

## 2. Data and build methods

Create the IFR datasets and build methods, with SOFR first and funding second.

In [ ]:
sofr_data = qfCreateData1D("INSTANTANEOUS FORWARD RATE", "SOFR-1B-IFR", sofr_quotes)
funding_data = qfCreateData1D(
    "INSTANTANEOUS FORWARD RATE", "SOFR-1B-FUNDING-IFR", funding_quotes,
)
data_collection = qfCreateDataCollection([sofr_data, funding_data])
display(data_collection.display())

In [ ]:
sofr_method = qfCreateBuildMethod("YIELD_CURVE_INDEX", {
    "TARGET": "SOFR-1B", "INSTANTANEOUS FORWARD RATE": "SOFR-1B-IFR",
})
funding_method = qfCreateBuildMethod("YIELD_CURVE_FUNDING", {
    "TARGET": FUNDING_INDEX, "REFERENCE INDEX": "SOFR-1B",
    "INSTANTANEOUS FORWARD RATE": "SOFR-1B-FUNDING-IFR",
})
build_methods = qfCreateModelBuildMethodCollection([sofr_method, funding_method])

## 3. Build the model

Copy IFR inputs into the model. The registered funding component references SOFR.

In [ ]:
model = qfCreateModel(VALUE_DATE, "YIELD_CURVE", data_collection, build_methods)
print("Component sizes:", model.num_sub_components)

The following helpers retrieve a component (`retrieve_model_component`), allocate zero-filled
gradient blocks in component order (`resize_gradient`), and locate a component's block
(`retrieve_model_component_index`, using the `component_indices` map).

In [ ]:
index = IndexRegistry().get("SOFR-1B")
component = cast(YieldCurveModelComponent, model.retrieve_model_component(index))
funding_component = model.retrieve_model_component(FUNDING_INDEX)

print("SOFR-1B ->", type(component).__name__, "|", component.component_identifier.name())
print("SOFR-1B-FLAT ->", funding_component.component_identifier.name())
display(pd.DataFrame(component.state_data.T, columns=["Node time (years)", "IFR"]))

In [ ]:
gradient = []
model.resize_gradient(gradient)
print("[SOFR block, funding block]:", gradient)

In [ ]:
print("Stored map:", model.component_indices)
print("SOFR-1B -> block", model.retrieve_model_component_index("SOFR-1B"))
print("SOFR-1B-FLAT -> block", model.retrieve_model_component_index(FUNDING_INDEX))

## 4. Discount-factor gradients

Complete **TODO 1** in `YieldCurveModelComponent` and **TODO 2** in `YieldCurve`
(`fixedincomelib/yield_curve/yield_curve_model.py`). Calculate the local derivative, then apply
the product rule recursively to $D_{\mathrm{funding}}=D_{\mathrm{spread}}D_{\mathrm{SOFR}}$.

Gradient methods write raw derivatives into the supplied container and return `None`.
`scaler` scales the new contribution; `accumulate=False` replaces the output and `True` adds to it.

With the default piecewise-constant interpolation, the third SOFR segment starts after
2027-01-02. Its gradient is zero for the 2026-10-02 payment.

In [ ]:
local_gradient = np.zeros(component.num_state_data)
component.discount_factor_gradient_wrt_state(PAYMENT_DATE, local_gradient)
print("Local SOFR discount-factor gradient:", local_gradient)

In [ ]:
model.discount_factor_gradient_wrt_state(FUNDING_INDEX, PAYMENT_DATE, gradient)
print("Model gradient [SOFR block, funding block]:", gradient)

## 5. Fixed-accrued valuation engine

Complete **TODOs 3–5** in `ValuationEngineProductFixedAccrued`: `__init__`, `calculate_value`,
and `calculate_risk` (`fixedincomelib/yield_curve/valuation_engine.py`).
Select the funding curve, then compute $C=N\alpha$, $PV=C D(T_p)$ and
$\nabla PV=C\nabla D(T_p)$. Notional is signed; this product has no separate fixed-rate field.

| Payment date | PV / cash today | Risk |
|---|---|---|
| Future | $C D(T_p)$ / 0 | $C\nabla D(T_p)$ |
| Today | $C$ / $C$ | 0 |
| Past | 0 / 0 | 0 |

In [ ]:
product = qfCreateProductFixedAccruedCashflow(
    "2026-04-02", "2026-07-02", "USD", 1_000_000, "ACT/360",
    payment_date=PAYMENT_DATE.ISO(), business_day_convention="NONE",
)
funding_parameter = qfCreateValuationParameters(
    "FUNDING INDEX PARAMETER", {"FUNDING INDEX": FUNDING_INDEX},
)
parameters = qfCreateValuationParametersCollection([funding_parameter])

print("Accrual fraction:", product.accrued)
print("Cash amount:", product.notional * product.accrued)

In [ ]:
engine = ValuationEngineProductFixedAccrued(model, parameters, product, ValuationRequest.PV)
engine.calculate_value()

product_gradient = []
engine.calculate_risk(product_gradient)

print("PV:", engine.value)
print("Cash today:", engine.cash)
print("Product gradient [SOFR block, funding block]:", product_gradient)

## 6. Reports

`qfCreateValueReport` calls the engine for `PV` or `FirstOrderRisk`.
`state_gradient` retains raw blocks; `display()` multiplies risk by `0.0001`
to show the signed PV change per positive 1 bp node shift.

In [ ]:
print("PV report:", qfCreateValueReport(model, product, parameters, "PV"))
risk_report = qfCreateValueReport(model, product, parameters, "FirstOrderRisk")

print("Raw blocks:", risk_report.state_gradient)
display(risk_report.display()[["COMPONENT", "NODE_INDEX", "AXIS1", "UNIT", "VALUES"]])